# 05 — Image Prompting and Visual Reasoning

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/05_image_prompting_and_visual_reasoning.ipynb)

**Prerequisites**: [01–04 multimodal](01_intro_to_multimodal_systems.ipynb), [PE/basic-prompt-structures](../prompt-engineering/basic-prompt-structures.ipynb)

**What you will learn**:
- Visual prompting patterns: direct, instructed, comparative, chain-of-thought
- Chart and graph analysis with structured prompts
- Screenshot and UI understanding
- Error analysis: common VLM failure modes

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Why Image Prompting Is Different**
- Understand **Chart and Graph Analysis**
- Understand **Error Analysis: When VLMs Fail**
- Understand **Exercises**
- Understand **Key Takeaways**


In [ ]:
# --- Setup: Load VLM ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow qwen-vl-utils

import json, math, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

plt.style.use("seaborn-v0_8-whitegrid")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/models"
except Exception:
    CACHE_DIR = "./models"

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Why Image Prompting Is Different

The model sees patches, not pixels. Prompt structure matters because it determines how the model allocates attention between text instructions and visual content. A well-structured prompt can double extraction accuracy.

Key patterns:
1. **Direct question**: "What is in this image?"
2. **Instructed analysis**: "Analyze the chart. Extract all data points as JSON."
3. **Comparative**: "Compare Image A and Image B. List differences."
4. **Chain-of-thought**: "First describe what you see. Then answer the question."

Each pattern activates different reasoning pathways in the model.

In [ ]:
# Visual prompting patterns comparison
patterns = {
    "Direct": "What is shown in this image?",
    "Instructed": "Analyze this image. List all objects with their colors and positions.",
    "Comparative": "Compare the left and right halves. What differs?",
    "Chain-of-thought": "First, describe all elements. Then, identify the main subject.",
}

for name, prompt in patterns.items():
    print(f"{name:20s}: {prompt}")
    print(f"{'':20s}  Token count: ~{len(prompt.split()):d} words")
    print()

print("In practice, instructed and CoT prompts yield more structured, accurate outputs.")
print("Direct prompts are fastest but least controllable.")

## Chart and Graph Analysis

Charts are one of the highest-value multimodal tasks: extracting numerical data from visual representations. The key challenge is getting **precise values** rather than qualitative descriptions.

Effective chart prompting:
- Tell the model what chart type to expect
- Ask for structured output (JSON or table)
- Request specific data points, not just "describe the chart"
- Validate extracted values against visual constraints

In [ ]:
# Chart analysis prompt engineering
chart_prompts = {
    "vague": "Describe this chart.",
    "structured": "This is a bar chart. Extract all category names and their values as a JSON list of {category, value} objects.",
    "validated": "This is a bar chart with values between 0 and 100. Extract each bar's label and value. Verify that values match the y-axis scale.",
}

for quality, prompt in chart_prompts.items():
    print(f"[{quality.upper()}] {prompt}")
    print()

print("The structured prompt gives the model a schema to follow.")
print("The validated prompt adds a sanity check constraint.")
print("Always prefer structured > vague for extraction tasks.")

## Error Analysis: When VLMs Fail

Common VLM failure modes:

| Failure type | Example | Frequency |
|-------------|---------|-----------|
| **Hallucinated text** | "Reading" text that isn't in the image | High |
| **Counting errors** | Saying 5 objects when there are 7 | High |
| **Spatial confusion** | "Left of" when it's "right of" | Medium |
| **Color misidentification** | Calling green "blue" in low contrast | Medium |
| **OCR errors** | Misreading characters in small text | Medium |

Understanding these failure modes helps you design better prompts and evaluation metrics.

In [ ]:
# Failure mode catalog
failures = pd.DataFrame([
    {"mode": "Hallucinated text", "risk": "High", "mitigation": "Ask model to quote exact text, verify with OCR"},
    {"mode": "Counting errors", "risk": "High", "mitigation": "Ask to list items individually before counting"},
    {"mode": "Spatial confusion", "risk": "Medium", "mitigation": "Use explicit coordinates or grid references"},
    {"mode": "Color misidentification", "risk": "Medium", "mitigation": "Describe colors relative to each other"},
    {"mode": "Small text OCR", "risk": "Medium", "mitigation": "Increase image resolution, crop relevant area"},
    {"mode": "Complex reasoning", "risk": "High", "mitigation": "Break into steps with chain-of-thought"},
])
print(failures.to_string(index=False))
print()
print("Key principle: Design prompts that work AROUND known failure modes.")
print("Test each failure type in your benchmark before deploying.")

## Exercises

1. **Adapt to your domain**: Apply the techniques from this notebook to a real task in your domain. Document what works and what doesn't.

2. **Error analysis**: Find 3 cases where the approach fails. Categorize the failure modes and propose fixes.

3. **Comparison**: Compare two different approaches from this notebook on the same test set. Which wins and why?

In [ ]:
# Exercise starter code
print("Replace this with your implementation.")
print("Document your findings in the markdown cells below.")

## Key Takeaways

This notebook covered the core concepts of **Image Prompting and Visual Reasoning**. The key principles are:

1. Start with the simplest approach that could work
2. Measure before and after every change
3. Understand failure modes to design robust pipelines
4. Budget tokens/compute before scaling up

## What's Next

Continue to the next notebook in the multimodal track.

## References

- See the Castalia multimodal README for the full reading list
- Relevant papers are cited inline throughout this notebook